In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re

In [36]:
url = 'https://www.amazon.in/gp/bestsellers/electronics/1805560031'

In [37]:
products = []

for page in range(1, 3):

    page_url = f'{url}?pg={page}'

    webpage = requests.get(page_url).text

    soup = BeautifulSoup(webpage, 'lxml')

    product_names = soup.find_all(
        'div',
        class_='_cDEzb_p13n-sc-css-line-clamp-3_g3dy1'
    )

    print("Page", page, ":", len(product_names))

    for product in product_names:
        products.append(product.text.strip())

print("Total products:", len(products))

Page 1 : 46
Page 2 : 44
Total products: 90


In [39]:
df = pd.DataFrame(products, columns=['Product'])
df.head()

,Product
0,"Samsung Galaxy M07 Mobile (Black, 4GB RAM, 64G..."
1,"Samsung Galaxy M36 5G Mobile (Velvet Black, 6G..."
2,"REDMI A7 Pro 5G (Mist Blue, 4GB RAM,128GB Stor..."
3,"iQOO Z10 Lite 5G (Cyber Green 2026, 4GB RAM, 6..."
4,"iQOO Z11 Lite 44W 5G (Solar Flame, 4GB RAM, 12..."


In [40]:
df.shape

(90, 1)

ACTUAL SEGREGATION PART 

In [41]:
df['Brand'] = df['Product'].str.split().str[0]
df[['Brand', 'Product']].head(10)

,Brand,Product
0,Samsung,"Samsung Galaxy M07 Mobile (Black, 4GB RAM, 64G..."
1,Samsung,"Samsung Galaxy M36 5G Mobile (Velvet Black, 6G..."
2,REDMI,"REDMI A7 Pro 5G (Mist Blue, 4GB RAM,128GB Stor..."
3,iQOO,"iQOO Z10 Lite 5G (Cyber Green 2026, 4GB RAM, 6..."
4,iQOO,"iQOO Z11 Lite 44W 5G (Solar Flame, 4GB RAM, 12..."
5,OnePlus,OnePlus N6 | 6GB+128GB | Midnight Green | Segm...
6,Samsung,"Samsung Galaxy A56 5G (Awesome Midnight Blue, ..."
7,Redmi,Redmi 15 5G Midnight Black 6GB + 128GB | Segme...
8,Motorola,"Motorola G57 Power 5G (Regatta, 8GB RAM, 128GB..."
9,OnePlus,OnePlus Nord CE6 Lite | 8GB+128GB | Hyper Blac...


In [42]:
df['Model'] = df['Product'].str.split('(').str[0].str.split('|').str[0].str.strip()

df['Model'] = df.apply(
    lambda row: row['Model'].replace(row['Brand'], '', 1).strip(),
    axis=1
)
df[['Brand', 'Model']].head(10)

,Brand,Model
0,Samsung,Galaxy M07 Mobile
1,Samsung,Galaxy M36 5G Mobile
2,REDMI,A7 Pro 5G
3,iQOO,Z10 Lite 5G
4,iQOO,Z11 Lite 44W 5G
5,OnePlus,N6
6,Samsung,Galaxy A56 5G
7,Redmi,15 5G Midnight Black 6GB + 128GB
8,Motorola,G57 Power 5G
9,OnePlus,Nord CE6 Lite


In [44]:
df['Color'] = df['Product'].str.extract(r'\(([^,]+),')[0]
df[['Brand', 'Model', 'Color']].head(10)

,Brand,Model,Color
0,Samsung,Galaxy M07 Mobile,Black
1,Samsung,Galaxy M36 5G Mobile,Velvet Black
2,REDMI,A7 Pro 5G,Mist Blue
3,iQOO,Z10 Lite 5G,Cyber Green 2026
4,iQOO,Z11 Lite 44W 5G,Solar Flame
5,OnePlus,N6,NaN
6,Samsung,Galaxy A56 5G,Awesome Midnight Blue
7,Redmi,15 5G Midnight Black 6GB + 128GB,NaN
8,Motorola,G57 Power 5G,Regatta
9,OnePlus,Nord CE6 Lite,NaN


In [45]:
df['Memory (in GB)'] = pd.to_numeric(
    df['Product'].str.extract(
        r'(\d+)\s*GB\s*(?:RAM)?',
        flags=re.IGNORECASE
    )[0],
    errors='coerce'
)
df[['Model', 'Memory (in GB)']].head(15)

,Model,Memory (in GB)
0,Galaxy M07 Mobile,4.0
1,Galaxy M36 5G Mobile,6.0
2,A7 Pro 5G,4.0
3,Z10 Lite 5G,4.0
4,Z11 Lite 44W 5G,4.0
5,N6,6.0
6,Galaxy A56 5G,8.0
7,15 5G Midnight Black 6GB + 128GB,6.0
8,G57 Power 5G,8.0
9,Nord CE6 Lite,8.0


In [46]:
df['Storage (in GB)'] = pd.to_numeric(
    df['Product'].str.extract(
        r'(?:RAM,\s*|RAM\s*|\+)(\d+)\s*GB(?:\s*Storage)?',
        flags=re.IGNORECASE
    )[0],
    errors='coerce'
)
df[['Model', 'Memory (in GB)', 'Storage (in GB)']].head(15)

,Model,Memory (in GB),Storage (in GB)
0,Galaxy M07 Mobile,4.0,64.0
1,Galaxy M36 5G Mobile,6.0,128.0
2,A7 Pro 5G,4.0,128.0
3,Z10 Lite 5G,4.0,64.0
4,Z11 Lite 44W 5G,4.0,128.0
5,N6,6.0,128.0
6,Galaxy A56 5G,8.0,NaN
7,15 5G Midnight Black 6GB + 128GB,6.0,NaN
8,G57 Power 5G,8.0,128.0
9,Nord CE6 Lite,8.0,128.0


In [47]:
ratings = []

for product in products:
    ratings.append(None)

In [48]:
product_cards = []

for page in range(1, 3):

    page_url = f'{url}?pg={page}'

    webpage = requests.get(page_url).text

    soup = BeautifulSoup(webpage, 'lxml')

    cards = soup.find_all(
        'div',
        attrs={'id': re.compile(r'gridItem')}
    )

    product_cards.extend(cards)

print("Total product cards:", len(product_cards))

Total product cards: 100


In [49]:
product_cards = product_cards[:len(df)]

print("Product cards:", len(product_cards))
print("DataFrame rows:", len(df))

Product cards: 90
DataFrame rows: 90


In [50]:
ratings = []

for card in product_cards:

    rating = card.find(string=re.compile(r'out of 5 stars'))

    if rating:
        rating = float(rating.split()[0])
    else:
        rating = None

    ratings.append(rating)

print("Ratings collected:", len(ratings))
print(ratings[:10])

Ratings collected: 90
[4.1, 4.1, 3.4, None, 4.1, 3.7, 3.9, 4.3, 4.0, 4.2]


In [51]:
df['Rating'] = ratings
df[['Brand', 'Model', 'Rating']].head(10)

,Brand,Model,Rating
0,Samsung,Galaxy M07 Mobile,4.1
1,Samsung,Galaxy M36 5G Mobile,4.1
2,REDMI,A7 Pro 5G,3.4
3,iQOO,Z10 Lite 5G,NaN
4,iQOO,Z11 Lite 44W 5G,4.1
5,OnePlus,N6,3.7
6,Samsung,Galaxy A56 5G,3.9
7,Redmi,15 5G Midnight Black 6GB + 128GB,4.3
8,Motorola,G57 Power 5G,4.0
9,OnePlus,Nord CE6 Lite,4.2


In [59]:
prices = []

for card in product_cards:

    price = card.find(string=re.compile(r'₹'))

    if price:
        price = re.search(r'₹([\d,]+)', price)
        price = int(price.group(1).replace(',', '')) if price else None
    else:
        price = None

    prices.append(price)

print("Prices collected:", len(prices))
print(prices[:10])

Prices collected: 90
[11999, 21999, 15999, 7999, 16999, 19999, 26999, 34999, 21499, 20998]


In [60]:
df['Price'] = prices

print(df[['Brand', 'Model', 'Color', 'Memory (in GB)', 'Storage (in GB)', 'Rating', 'Price']].head())

     Brand                 Model             Color  Memory (in GB)  \
0  Samsung     Galaxy M07 Mobile             Black             4.0   
1  Samsung  Galaxy M36 5G Mobile      Velvet Black             6.0   
2    REDMI             A7 Pro 5G         Mist Blue             4.0   
3     iQOO           Z10 Lite 5G  Cyber Green 2026             4.0   
4     iQOO       Z11 Lite 44W 5G       Solar Flame             4.0   

   Storage (in GB)  Rating    Price  
0             64.0     4.1  11999.0  
1            128.0     4.1  21999.0  
2            128.0     3.4  15999.0  
3             64.0     NaN   7999.0  
4            128.0     4.1  16999.0  


In [62]:
df = df[
    ['Brand',
     'Model',
     'Color',
     'Memory (in GB)',
     'Storage (in GB)',
     'Rating',
     'Price']
]

print(df.head(40))
print("\nShape:", df.shape)

       Brand                                              Model  \
0    Samsung                                  Galaxy M07 Mobile   
1    Samsung                               Galaxy M36 5G Mobile   
2      REDMI                                          A7 Pro 5G   
3       iQOO                                        Z10 Lite 5G   
4       iQOO                                    Z11 Lite 44W 5G   
5    OnePlus                                                 N6   
6    Samsung                                      Galaxy A56 5G   
7      Redmi                   15 5G Midnight Black 6GB + 128GB   
8   Motorola                                       G57 Power 5G   
9    OnePlus                                      Nord CE6 Lite   
10   OnePlus                                             Nord 6   
11  Motorola                                       G57 Power 5G   
12   Samsung                               Galaxy M17 5G Mobile   
13      Lava                                            Bold N

In [63]:
df.head()

,Brand,Model,Color,Memory (in GB),Storage (in GB),Rating,Price
0,Samsung,Galaxy M07 Mobile,Black,4.0,64.0,4.1,11999.0
1,Samsung,Galaxy M36 5G Mobile,Velvet Black,6.0,128.0,4.1,21999.0
2,REDMI,A7 Pro 5G,Mist Blue,4.0,128.0,3.4,15999.0
3,iQOO,Z10 Lite 5G,Cyber Green 2026,4.0,64.0,NaN,7999.0
4,iQOO,Z11 Lite 44W 5G,Solar Flame,4.0,128.0,4.1,16999.0


In [64]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Brand            90 non-null     object 
 1   Model            90 non-null     str    
 2   Color            64 non-null     str    
 3   Memory (in GB)   88 non-null     float64
 4   Storage (in GB)  60 non-null     float64
 5   Rating           88 non-null     float64
 6   Price            85 non-null     float64
dtypes: float64(4), object(1), str(2)
memory usage: 5.1+ KB


In [65]:
df.to_csv('amazon_smartphones.csv', index=False)

print("CSV file created")

CSV file created successfully!


In [66]:
print(df[df['Color'].str.contains(r'GB', case=False, na=False)])

      Brand                       Model Color  Memory (in GB)  \
40     POCO    M7 Plus 5G, Carbon Black   4GB             4.0   
52  Samsung  Galaxy F06 5G, Bahama Blue   4GB             4.0   
78     Poco         C81X, Crystal Black   3GB             3.0   
83  Samsung        Galaxy F36 5G, Black   6GB             6.0   

    Storage (in GB)  Rating    Price  
40              NaN     3.3   8599.0  
52              NaN     3.5  39999.0  
78              NaN     4.1  35799.0  
83              NaN     4.0  41999.0  


In [71]:
for i in [40, 52, 78, 83]:

    product = products[i]

    main_part = product.split('(')[0].strip()

    if ',' in main_part:
        model_part, color = main_part.split(',', 1)
        color = color.strip()
    else:
        model_part = main_part
        color = None

    brand = df.loc[i, 'Brand']
    model = model_part.replace(brand, '', 1).strip()

    specs = re.search(r'\((\d+)\s*GB,\s*(\d+)\s*GB\)', product)

    if specs:
        memory = int(specs.group(1))
        storage = int(specs.group(2))
    else:
        memory = None
        storage = None

    df.loc[i, 'Model'] = model
    df.loc[i, 'Color'] = color
    df.loc[i, 'Memory (in GB)'] = memory
    df.loc[i, 'Storage (in GB)'] = storage

print(df.loc[[40, 52, 78, 83]])

      Brand          Model          Color  Memory (in GB)  Storage (in GB)  \
40     POCO     M7 Plus 5G   Carbon Black             4.0            128.0   
52  Samsung  Galaxy F06 5G    Bahama Blue             4.0            128.0   
78     Poco           C81X  Crystal Black             3.0             64.0   
83  Samsung  Galaxy F36 5G          Black             6.0            128.0   

    Rating    Price  
40     3.3   8599.0  
52     3.5  39999.0  
78     4.1  35799.0  
83     4.0  41999.0  


In [72]:
df['Memory (in GB)'] = df['Memory (in GB)'].astype('Int64')
df['Storage (in GB)'] = df['Storage (in GB)'].astype('Int64')
df['Price'] = df['Price'].astype('Int64')

print(df.head())
print("\nData types:")
print(df.dtypes)

     Brand                 Model             Color  Memory (in GB)  \
0  Samsung     Galaxy M07 Mobile             Black               4   
1  Samsung  Galaxy M36 5G Mobile      Velvet Black               6   
2    REDMI             A7 Pro 5G         Mist Blue               4   
3     iQOO           Z10 Lite 5G  Cyber Green 2026               4   
4     iQOO       Z11 Lite 44W 5G       Solar Flame               4   

   Storage (in GB)  Rating  Price  
0               64     4.1  11999  
1              128     4.1  21999  
2              128     3.4  15999  
3               64     NaN   7999  
4              128     4.1  16999  

Data types:
Brand               object
Model                  str
Color                  str
Memory (in GB)       Int64
Storage (in GB)      Int64
Rating             float64
Price                Int64
dtype: object


In [74]:
df.to_csv('amazon_smartphones_cleaned.csv', index=False)

print("Cleaned CSV saved successfully!")

Cleaned CSV saved successfully!
